In [1]:
!rm -f dndmsc26spring_hw1_tester.py 
!curl -O http://nipg12.inf.elte.hu/~vavsaai@nipg.lab/dndmsc26spring/files/dndmsc26spring_hw1_tester.py

import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from dndmsc26spring_hw1_tester import Tester

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14757  100 14757    0     0   116k      0 --:--:-- --:--:-- --:--:--  117k


In [2]:
tester = Tester()
content = tester.get_dataset_content()

print("Number of characters in dataset:", len(content))
print(content[:500])

Number of characters in dataset: 27418
school;sex;age;address;famsize;Pstatus;Medu;Fedu;Mjob;Fjob;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3
0;0;18;0;1;1;4;4;5;0;4;;2;2;0;0;1;1;1;0;0;1;1;4;3;4;1;1;3;6;14;15;15
0;0;17;0;1;0;1;1;5;3;4;1;1;2;0;1;0;1;1;1;0;0;1;5;3;3;1;1;3;4;6;6;7
0;0;15;0;0;0;1;1;5;;3;;1;2;3;0;1;0;1;0;0;0;1;4;3;2;2;3;3;10;7;7;8
0;0;15;0;1;0;4;2;1;4;0;;1;3;0;1;0;0;0;0;0;0;0;3;2;2;1;1;5;2;15;15;15
0


A. Loading the Dataset

In [3]:
lines = content.splitlines()

data_lines = lines[1:]

data = []

for line in data_lines:
    values = line.split(';')
    row = []

    for i, val in enumerate(values):
        if val == "":
            if i in [8, 9, 19]:
                row.append(np.nan)
            else:
                row.append(np.nan)
        else:
            row. append(np.float32(val))

    data.append(row)

dataset_noisy = np.array(data, dtype=np.float32)

tester.test('dataset_load', dataset_noisy)

Tester: Dataset loading OK


B. Handling the Missing Data

In [10]:
nan_columns = [8, 9, 11]

indicators = []

for col in nan_columns:
    indicator = (~np.isnan(dataset_noisy[:, col])).astype(np.float32)
    indicators.append(indicator.reshape(-1, 1))

indicator_matrix = np.hstack(indicators)

dataset = np.hstack([dataset_noisy, indicator_matrix])

dataset = np.nan_to_num(dataset, nan=0.0)

tester.test('dataset_fill_missing', dataset)

Tester: Dataset fill missing OK


## C: Splitting into training, validation, and test sets

In [ ]:
np.random.seed(42)
shuffled_indices = np.random.permutation(dataset.shape[0])
dataset_shuffled = dataset[shuffled_indices]

n = dataset.shape[0]
train_end = int(0.5 * n)
val_end  =  int(0.75 * n)

dataset_split_train = dataset_shuffled[:train_end]
dataset_split_val = dataset_shuffled[train_end:val_end]
dataset_split_test = dataset_shuffled[val_end:]

tester.test('dataset_split', dataset_split_train, dataset_split_val, dataset_split_test)

Tester: Dataset split OK
<bound method Tester.test of <dndmsc26spring_hw1_tester.Tester object at 0x1126517f0>>


## D: Creating data iterators for the regression task

In [14]:
class StudentDataset(Dataset):
    def __init__(self, data_array):
        self.X = np.delete(data_array, 32, axis=1)
        self.y = data_array[:, 32].reshape(-1, 1)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        x = torch.tensor(self.X[index], dtype=torch.float32)
        y = torch.tensor(self.y[index], dtype=torch.float32)
        return x, y
    
train_dataset = StudentDataset(dataset_split_train)
val_dataset = StudentDataset(dataset_split_val)
test_dataset = StudentDataset(dataset_split_test)

batchsize = 32

dataloader_reg_train = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
dataloader_reg_val = DataLoader(val_dataset, batch_size=batchsize, shuffle=True)
dataloader_reg_test = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

tester.test('reg_iter', dataloader_reg_train, dataloader_reg_val, dataloader_reg_test)



Tester: Dataset iterators for regression task OK


## E: Defining the regression neural network

In [15]:
class RegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(35, 25)
        self.fc2 = nn.Linear(25, 25)
        self.fc3 = nn.Linear(25, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
reg_model = RegressionModel()

tester.test('reg_model_architecture', reg_model)

Tester: Regression model architecture OK
